<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Far from the neighbours: kNN distance

**[Far from the neighbours: kNN distance](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/)** · Anomaly Detection: Find What Is Unusual · Module 4, lesson 1 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score a payment by its distance to its nearest neighbours among normal payments, and see why scaling and the choice of k change the result.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. The notebook takes about a minute to run. |
| **You should know** | The comparison table and the reference and evaluation months, from [Meet the Larkfield data](https://learning.nextia-ai.com/courses/anomaly-detection/m01/meet-the-larkfield-data/); scaling, from the machine learning course. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M04-L01-far-from-the-neighbours-knn-distance/knn-distance-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Far from everybody: a tiny example

At a party, most guests stand in groups. One guest stands alone in a corner. You do not need to know anything about the guests: the one far from the others is unusual.

The **k-nearest-neighbours distance** (kNN distance) uses this idea. For each point, find its k nearest neighbours; the distance to the k-th one is the score. A point in a crowd has a small score; a point alone has a large one.

Run the next cell. Seven points on a plane: six close together, and G far away. It prints each point's distance to its 2nd nearest neighbour (k = 2).

> **Check.** You should see G = 2.62 and every other point at 0.22 to 0.32.

In [ ]:
points = np.array([[1.0, 1.0], [1.2, 0.9], [0.9, 1.1], [1.1, 1.2], [0.8, 0.8], [1.0, 1.3], [3.0, 3.0]])
names = ["A", "B", "C", "D", "E", "F", "G"]
distances = np.sqrt(((points[:, None, :] - points[None, :, :]) ** 2).sum(axis=2))   # every pair
np.fill_diagonal(distances, np.inf)                     # a point is not its own neighbour
k = 2
kth = np.sort(distances, axis=1)[:, k - 1]               # the distance to the 2nd nearest neighbour
print(pd.Series(kth.round(2), index=names).to_string())

## 2. Larkfield's payments, raw features

The detector learns "normal" from a sample of 20,000 payments of the reference months, and scores every payment of the evaluation months. First, the raw features: the amount (as a logarithm, so that 10 and 1,000 dollars are not 100 times apart), the hour and the account type.

In [ ]:
import compare, features
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
raw = features.raw(payments)
raw["amount"] = np.log(raw.amount)                      # log: 10 -> 2.3, 1,000 -> 6.9
print(raw.loc[evaluation.index].head(3))

Run the next cell. It defines `knn_scores()` (scale, then the distance to the k-th neighbour) and adds the first row.

> **Check.** You should see a precision at budget of 0.219.

In [ ]:
def knn_scores(X, k=10, sample=20_000, seed=0):
    """Distance of each evaluation payment to its k-th nearest neighbour among reference payments."""
    scaler = StandardScaler().fit(X.loc[reference.index])
    ref = scaler.transform(X.loc[reference.index])
    ref = ref[np.random.default_rng(seed).choice(len(ref), sample, replace=False)]   # a sample: kNN is slow
    nn = NearestNeighbors(n_neighbors=k).fit(ref)
    dist, _ = nn.kneighbors(scaler.transform(X.loc[evaluation.index]))
    return dist[:, -1]

table = compare.Table()
table.add(compare.score_payments(evaluation, knn_scores(raw), "kNN distance", "raw features, k = 10", explain_minutes=3))
table.show()

## 3. Predict, then run: does scaling matter?

The amount in dollars goes from 1 to several thousands; the hour from 0 to 23; the account type is 0 or 1. **Scaling** puts every feature on the same scale (mean 0, standard deviation 1).

> **Predict.** Without scaling and without the logarithm, will kNN find more, the same, or fewer planted frauds?

In [ ]:
unscaled = features.raw(payments)                        # amount in dollars, hour 0-23, business 0/1
ref = unscaled.loc[reference.index].to_numpy()[np.random.default_rng(0).choice(len(reference), 20_000, replace=False)]
dist, _ = NearestNeighbors(n_neighbors=10).fit(ref).kneighbors(unscaled.loc[evaluation.index].to_numpy())
row = compare.score_payments(evaluation, dist[:, -1], "kNN distance, not scaled", "raw dollars and hours")
print("not scaled:", row["precision at budget"], "| scaled:", table.rows[0]["precision at budget"])

> **Check.** You should see `not scaled: 0.013`.

Without scaling, the distance is almost only the amount in dollars: an hour of 3 a.m. against 3 p.m. is a difference of 12, a basket of 40 against 52 dollars too. Distance methods need features on comparable scales, and with mixed feature types (amounts, hours, yes/no) you must choose them with care.

## 4. Change one thing: behaviour features

The raw columns do not say what makes a stolen card strange. Module 4's `features.py` compares each payment with the customer's own past: the amount against the customer's usual one, a new country, a new device, the payments in the last hour, the payments on the device in the last hour, the night. Lesson 5 of this module builds them step by step. Run the next cell.

> **Check.** You should see a precision at budget of 0.505 with the behaviour features.

In [ ]:
behaviour = features.behaviour(payments)
print(behaviour.columns.tolist())
table.add(compare.score_payments(evaluation, knn_scores(behaviour), "kNN distance", "behaviour features, k = 10",
                                 explain_minutes=3))
table.show()

The same algorithm went from 0.219 to 0.505: the features changed more than any setting would. Keep this in mind for the whole module.

## 5. A common mistake: the wrong k

With k = 1, a point next to one other strange point (two payments of the same stolen card) looks normal: its nearest neighbour is close. With a large k, small groups of frauds are found, but the score gets slower and smoother.

## 6. Check yourself

**Your turn 1.** In the tiny example, what is point G's distance to its 2nd nearest neighbour? Write it as `g_distance` (2 decimals).

In [ ]:
g_distance = 2.62

In [ ]:
expect('g_distance', g_distance, 2.62, tolerance=0.01)

**Your turn 2.** Score the behaviour features with k = 1, 10 and 50. Put each precision at budget in a dictionary `by_k` (k → precision), and the best k in `best_k`. This cell takes about 20 seconds.

In [ ]:
by_k = {k: compare.score_payments(evaluation, knn_scores(behaviour, k=k), "kNN", f"k = {k}")["precision at budget"]
        for k in (1, 10, 50)}
best_k = max(by_k, key=by_k.get)
print(by_k)

In [ ]:
expect_hash('best_k', best_k, '1a6562590ef19d10')

In the solution: k = 1: 0.450, k = 10: 0.505, k = 50: 0.515.

**Your turn 3.** Add your best k as a row of `table` with the setting `"behaviour features, best k"`, and show the table. (No check: compare it with the rows above.)

In [ ]:
table.add(compare.score_payments(evaluation, knn_scores(behaviour, k=best_k), "kNN distance", "behaviour features, best k",
                                 explain_minutes=3))
table.show()

## Recap

- kNN distance: a payment far from its nearest normal neighbours is unusual. It needs no labels, only normal data.
- Scaling decides what "far" means: without it, the dollars decided everything (0.013).
- Better features gave the largest gain (0.219 → 0.505). kNN is slow on many rows: here it compares each payment with a sample of 20,000.

The next lesson asks a sharper question: far from the neighbours, **for this area**?

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Local density: LOF](https://learning.nextia-ai.com/courses/anomaly-detection/m04/local-density-lof/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m04/far-from-the-neighbours-knn-distance/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.